In [ ]:
# Online MBE computation
# ----------------------------
import torch
from src.mbe import mbe_reverse_gram
B = 4
L = 4096 
D = 256 
patch_size = 8
R = torch.randn(B, L, D)

from src.mbe import OnlineMBE

online_mbe = OnlineMBE(D=D)
for t in range(L): 
    online_mbe.update(R[:, t, :])

print(f"G shape: {online_mbe.G.shape}")  # (D, D)
print(f"MBE: {online_mbe.mbe().item():.4f}")  # scalar

G shape: torch.Size([256, 256])
MBE: 5.5296


In [ ]:
# CE + lambda_mbe

from src.ibrl import IBRLTrainer, IBRLConfig
from datasets import Dataset
import torch
 
def reward_fn(prompts, completions, **kwargs):
    return [1.0 for _ in completions]
 
ds = Dataset.from_dict({'prompt': ['What is 2+3? Answer:'] * 8})
 
config = IBRLConfig(
    output_dir='/tmp/ibrl_test_debug2',
    num_generations=2,
    max_completion_length=16,
    temperature=0.8,
    beta=0.1,
    lambda_mbe=0.01,
    mbe_layer=-1,
    mbe_patch_size=8,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=1e-6,
    logging_steps=1,
    report_to='none',
    log_completions=False,
    bf16=False,
    model_init_kwargs={'torch_dtype': torch.float32},
)
 
trainer = IBRLTrainer(
    model='Qwen/Qwen2.5-0.5B-Instruct',
    reward_funcs=reward_fn,
    args=config,
    train_dataset=ds,
)
 
trainer.train()
print('SUCCESS')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,-0.011388
2,0.106053
3,0.005300
4,0.019090


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS
